# EMMa Dataset Filtering

In [ ]:
import json
from pathlib import Path
import numpy as np

from emma import EmmaDataset, EmmaSearcher, SearchQuery

In [ ]:
DATA_DIR = Path().resolve().parent.parent
DATA_CONFIG = DATA_DIR / "configs" / "data.json"
with DATA_CONFIG.open(encoding="utf-8") as f:
    data_config = json.load(f)

DATASET_ROOT = Path(data_config["emma_root"])
GLP_DEPTH_PATH = Path(data_config["valid_asins"]["glp"])
GLP_DEPTH_CORRECTED_PATH = Path(data_config["valid_asins"]["glp_corrected"])
DEPTH_ANYTHING_PATH = Path(data_config["valid_asins"]["depth_anything_v3"])
THRESHOLDING_PATH = Path(data_config["valid_asins"]["thresholding"])
SAM3_PATH = Path(data_config["valid_asins"]["sam3"])
CLIP_PATH = Path(data_config["valid_asins"]["clip"])

PRE_AUG_ASINS_PATH = Path(data_config["asins"]["pre_augmentation"])
POST_AUG_ASINS_PATH = Path(data_config["asins"]["post_augmentation"])

In [ ]:
ds = EmmaDataset(DATASET_ROOT)
se = EmmaSearcher(ds)

### 1. PRE-AUGMENTATION FILTERING

In [ ]:
# Categories to exclude
excluded_categories = [
    (23, 0),  # software
    (22, 0),  # kindle store
    (16, 0),  # books
    (17, 0),  # video games
    (20, 0),  # movies & tv
    (19, 0),  # cds & vinyl
    (24, 0),  # gift cards
]

excluded_asins = set()
for key in excluded_categories:
    excluded_asins.update(se.query_asins(SearchQuery(categories=[key])))

print("ASINs to exclude:", len(excluded_asins))

In [ ]:
def print_filter_summary(split_name=None):
    label = "ALL" if split_name is None else split_name.upper()
    query = SearchQuery(splits=[split_name]) if split_name else SearchQuery()

    split_asins = se.query_asins(query)
    mass_asins = se.query_asins(query, require_listed=["mass"])
    dimension_asins = se.query_asins(
        query, require_listed=["mass"], require_present=["package_dimensions"]
    )
    single_item_asins = se.query_asins(
        query,
        require_listed=["mass"],
        require_present=["package_dimensions"],
        quantity_range=[1, 1],
    )
    categorized_asins = se.query_asins(
        query,
        require_listed=["mass", "category"],
        require_present=["package_dimensions"],
        quantity_range=[1, 1],
    )

    print(label)
    print(f"{'Splits':<24} ASINs: {len(split_asins):,}")
    print(f"{'Mass listed':<24} ASINs: {len(mass_asins):,}")
    print(f"{'Dimensions':<24} ASINs: {len(dimension_asins):,}")
    print(f"{'Quantity = 1':<24} ASINs: {len(single_item_asins):,}")
    print(f"{'Categories listed':<24} ASINs: {len(categorized_asins):,}")
    print(
        f"{'Excluding categories':<24} ASINs: "
        f"{sum(asin not in excluded_asins for asin in categorized_asins):,}"
    )
    print()

for split_name in [None, "Train", "Val", "Test"]:
    print_filter_summary(split_name)

In [ ]:
query = SearchQuery()
pre_asins = [
    asin
    for asin in se.query_asins(
        query,
        require_listed=["mass", "category"],
        require_present=["package_dimensions"],
        quantity_range=[1, 1],
    )
    if asin not in excluded_asins
]
pre_asins = np.asarray(pre_asins, dtype=str)

In [ ]:
PRE_AUG_ASINS_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(PRE_AUG_ASINS_PATH, pre_asins)
print(f"Saved {len(pre_asins)} ASINs to: {PRE_AUG_ASINS_PATH}")

### 2. POST-AUGMENTATION FILTERING

In [ ]:
def load_npy(path):
    return np.load(path, allow_pickle=False)

In [ ]:
asin_sets = {
    "GLP Depth": set(load_npy(GLP_DEPTH_PATH)),
    "GLP Depth Corrected": set(load_npy(GLP_DEPTH_CORRECTED_PATH)),
    "Depth Anything": set(load_npy(DEPTH_ANYTHING_PATH)),
    "Thresholding Segmentation": set(load_npy(THRESHOLDING_PATH)),
    "SAM3 Segmentation": set(load_npy(SAM3_PATH)),
    "CLIP": set(load_npy(CLIP_PATH)),
}

augmented_asins = set.intersection(*asin_sets.values())

for name, asins in asin_sets.items():
    print(f"{name:<24} ASINs: {len(asins):,}")

print(f"{'All augmentations':<24} ASINs: {len(augmented_asins):,}")

In [ ]:
post_asins = [
    asin
    for asin in pre_asins
    if asin in augmented_asins
]
post_asins = np.asarray(post_asins, dtype=str)

if not len(post_asins):
    raise RuntimeError("Post-augmentation filtering selected no ASINs.")

print(f"Pre-augmentation ASINs: {len(pre_asins)}")
print(f"Excluded after augmentation: {len(pre_asins) - len(post_asins)}")

In [ ]:
post_asin_set = set(post_asins.tolist())
split_counts = {"All": len(post_asin_set)}

for split_name in ["Train", "Val", "Test"]:
    split_asins = set(se.query_asins(SearchQuery(splits=[split_name])))
    split_counts[split_name] = len(post_asin_set & split_asins)

print("Post-augmentation ASIN counts:")
for split_name, count in split_counts.items():
    print(f"{split_name:<24} ASINs: {count:,}")

In [ ]:
POST_AUG_ASINS_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(POST_AUG_ASINS_PATH, post_asins)
print(f"Saved {len(post_asins)} ASINs to: {POST_AUG_ASINS_PATH}")